# Phần 3 — R-GAP trên ResNet-18: Rank Analysis + Inverting Gradients

**Pillar:** Security — privacy trong Federated Learning (FL).

**Một hiểu lầm cần làm rõ trước.** Đọc lại kỹ paper R-GAP (Zhu & Blaschko, ICLR 2021, Section 5 "Results" + Figure 4), thí nghiệm ResNet-18 của chính tác giả **không chạy thuật toán đệ quy `r_gap`** (như `part3.ipynb`/`part3_batch2.ipynb` làm trên LeNet) — nguyên văn:

> *"To perform the experiments, we use the approach proposed by Geiping et al. (2020) [Inverting Gradients], which extends DLG to incorporate image priors and performs better on deep networks."*

Lý do: `r_gap` (bản cài đặt trong repo gốc) cần activation invertible từng-phần-tử và không xử lý skip-connection/BatchNorm — không áp dụng trực tiếp lên ResNet được. Thay vào đó, paper dùng **hai công cụ khác, cả hai đều khả thi để tái hiện**:

1. **Rank Analysis (RA-i)** — một công thức đếm chiều (không cần optimize, không cần chạy attack), dùng để *dự đoán* layer/block nào dễ bị tấn công nhất, chỉ từ kiến trúc.
2. **Inverting Gradients** (Geiping et al., 2020) — **chính là thuật toán Part 2 đã có sẵn** (`inversefed.GradientReconstructor`) — dùng để *verify* dự đoán đó bằng cách thật sự attack 3 biến thể ResNet-18: bản gốc, bản cắt skip-connection ở block "critical" (dự đoán bởi RA-i), và bản cắt skip-connection ở một block không quan trọng (control).

Notebook này tái hiện đúng quy trình đó, không phải một mở rộng tự chế của thuật toán `r_gap`.

**Kiến trúc.** Paper dùng ResNet-18 dạng CIFAR: `conv3x3@16` (stem) → 8 residual block (base width 16, tăng dần 16→32→64→128, downsample ở block 3/5/7) → avgpool → fc(128,10), huấn luyện trên CIFAR-10. Kiến trúc này **đã có sẵn** trong `inversefed/nn/models.py` (chính package Geiping et al. dùng cho các thí nghiệm của họ, có BatchNorm) — chỉ cần gọi `ResNet(BasicBlock, [2,2,2,2], base_width=16)`.

**Những gì tôi tự thu hẹp so với paper (nêu rõ, không giấu):**
- **RA-i chỉ tính "local"** (công thức chính trong Algorithm 1 + Appendix D: $|x_i| - |W_i| - |z_i|$, và với residual block $|x_1| - |W_2| - |f|$), **không cộng dồn "virtual constraints"** từ các layer trước (công thức đầy đủ ở Appendix C, phụ thuộc dữ liệu/trọng số cụ thể, phức tạp hơn nhiều để tái hiện chính xác). Paper cũng dùng bản local này cho phần lớn các ví dụ minh hoạ (Figure 2, cột 1-3), nên đây là một sự thu hẹp hợp lý, không phải bỏ qua phần cốt lõi.
- **Huấn luyện 15 epoch**, không phải 200 epoch như paper — đủ để 3 biến thể đạt **accuracy gần nhau** (điều paper muốn chứng minh: cắt skip-connection không nhất thiết hại accuracy) nhưng không đạt SOTA.
- **1 lần thử** (1 ảnh, 1 lần train mỗi biến thể), không lấy trung bình nhiều lần chạy như paper.
- Attack dùng **đúng ngân sách đầy đủ của `part2.ipynb` mục 4** (24 000 iteration, 2 restart, `filter='median'`) — thử ở ngân sách nhỏ hơn (3 000 iteration, 1 restart) trước và **cả 3 biến thể đều ra nhiễu thuần túy** (PSNR ~11 dB, không phân biệt được với nhau) dù model đã train tới ~86% accuracy: model đã train khó attack hơn hẳn model random-init (đúng như Geiping et al. và chính paper R-GAP đều lưu ý), nên cần ngân sách attack đầy đủ mới mong thấy được khác biệt giữa các biến thể.

In [ ]:
import sys, os, time, copy, types
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))   # repo root, so `import inversefed` resolves

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.models.resnet import BasicBlock
import matplotlib.pyplot as plt

import inversefed
import inversefed.data.data_processing as _dp
_dp.MULTITHREAD_DATAPROCESSING = 0   # avoid a Windows spawn-multiprocessing crash in DataLoader workers (not an algorithmic issue)

torch.manual_seed(50)
np.random.seed(50)

setup = inversefed.utils.system_startup()
print(setup)
print(torch.__version__, torchvision.__version__)

## 1. Kiến trúc — CIFAR ResNet-18, base width 16

Tái dùng `inversefed.nn.models.ResNet` (đúng class Geiping et al. dùng cho các thí nghiệm CIFAR của chính họ) với `BasicBlock` của torchvision — 8 block, mỗi block 2 conv layer + skip-connection, tổng 18 layer có trọng số (16 conv trong block + stem conv + fc) — khớp đúng mô tả Figure 4 của paper R-GAP.

In [ ]:
def build_resnet18_cifar(base_width=16, num_classes=10):
    return inversefed.nn.models.ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes, base_width=base_width)

def get_blocks(model):
    # flatten the 4 stages into an ordered list of 8 BasicBlocks (block 1..8, matching the paper's numbering)
    return [blk for stage in model.layers for blk in stage]

base_model = build_resnet18_cifar()
blocks = get_blocks(base_model)
print("num residual blocks:", len(blocks))
for i, b in enumerate(blocks, 1):
    print("block %d: %s -> %s %s" % (i, tuple(b.conv1.weight.shape[1:]), tuple(b.conv2.weight.shape[:1]) + tuple(b.conv1.weight.shape[2:]),
                                     "(downsample)" if b.downsample is not None else ""))

## 2. Rank Analysis Index (RA-i) — dự đoán block nào dễ bị attack nhất

Từ Algorithm 1 và Appendix D của paper, RA-i "local" (bỏ qua virtual constraints) tính bằng:

- **Layer thường** (vd. stem conv): $\mathrm{RA\text{-}i} = |x_i| - |W_i| - |z_i|$ — số ẩn số ($x_i$ = input) trừ số ràng buộc từ gradient ($|W_i|$ = số phần tử weight) và từ forward pass ($|z_i|$ = số phần tử output).
- **Residual block (skip qua 2 layer)** — Appendix D, Eq. 29-32: vì $f = W_2 x_2 + x_1$ và $x_2 = W_1 x_1$, sau khi viết lại dạng tương đương, chỉ cần $|W_2| + |f|$ ràng buộc để suy ra **toàn bộ input của block** $x_1$ (input của conv1, TRƯỚC khi qua conv1) — không cần đến $|W_1|$ hay $|x_2|$ riêng:

$$\mathrm{RA\text{-}i}_{\text{block}} = |x_1| - |W_2| - |f|$$

**RA-i càng lớn (càng gần 0 hoặc dương) → block càng khó bị recover đầy đủ → càng "critical" theo nghĩa ngược: đây là layer làm attack khó, nên nếu ta phá vỡ khả năng "tự bù" của skip-connection ở chính layer này, defendability tăng mạnh nhất.** (Cách paper dùng: tìm block có RA-i lớn nhất trong số các block *có* skip connection, vì đó là nơi skip-connection đang "gánh" nhiều nhất cho việc reconstruct; cắt nó sẽ làm mất đúng phần bù đó.)

In [ ]:
def numel(shape):
    n = 1
    for d in shape:
        n *= d
    return n

def ra_index_conv(in_shape, weight_shape, out_shape):
    return numel(in_shape) - numel(weight_shape) - numel(out_shape)

def ra_index_block(in_shape, w2_shape, out_shape):
    return numel(in_shape) - numel(w2_shape) - numel(out_shape)

# trace true input/output shapes of the stem conv and every block with forward hooks
shapes_in, shapes_out = {}, {}
def hook(name):
    def _hook(mod, inp, out):
        shapes_in[name] = tuple(inp[0].shape[1:])
        shapes_out[name] = tuple(out.shape[1:])
    return _hook

handles = [base_model.conv1.register_forward_hook(hook('stem'))]
for i, b in enumerate(blocks, 1):
    handles.append(b.register_forward_hook(hook('block%d' % i)))
base_model.eval()
with torch.no_grad():
    base_model(torch.randn(1, 3, 32, 32))
for h in handles:
    h.remove()

names, ra_values, is_downsample = ['stem'], [ra_index_conv(shapes_in['stem'], base_model.conv1.weight.shape, shapes_out['stem'])], [False]
for i, b in enumerate(blocks, 1):
    name = 'block%d' % i
    ra = ra_index_block(shapes_in[name], b.conv2.weight.shape, shapes_out[name])
    names.append(name); ra_values.append(ra); is_downsample.append(b.downsample is not None)
    print("%-8s in=%-16s out=%-14s RA-i=%8d %s" % (name, shapes_in[name], shapes_out[name], ra, "<= has skip-connection" if i >= 1 else ""))

critical_block = max(range(1, 9), key=lambda i: ra_values[i])   # largest RA-i among the 8 blocks
print("\n-> most critical residual block (largest RA-i): block %d" % critical_block)

plt.figure(figsize=(8, 3.2))
colors = ['gray'] + ['crimson' if i == critical_block else ('orange' if ds else 'steelblue') for i, ds in zip(range(1, 9), is_downsample[1:])]
plt.bar(names, ra_values, color=colors)
plt.ylabel("RA-i (local)")
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.title("Rank Analysis Index per layer (red = most critical block, orange = other downsample blocks)")
plt.tight_layout()
plt.show()

## 3. Ba biến thể — gốc, cắt skip-connection critical, cắt skip-connection control

Theo đúng thiết kế Figure 4 của paper: **Variant 1** cắt skip-connection của block critical (dự đoán ở mục 2 — nếu quy trình đúng, phải ra **block 3**, đúng như paper báo cáo); **Variant 2** cắt skip-connection của **block 8** (block cuối, không downsample) làm control — dự đoán sẽ gần như không ảnh hưởng gì. Cắt skip-connection = bỏ dòng `out += identity` trong `forward` của `BasicBlock` (tương đương phân tích ở Appendix D: *"this is the same as removing the skip connection"*). Cả 3 model xuất phát từ **cùng một bộ trọng số khởi tạo** (deepcopy trước khi patch forward) để so sánh công bằng.

In [ ]:
def forward_no_skip(self, x):
    out = self.conv1(x); out = self.bn1(out); out = self.relu(out)
    out = self.conv2(out); out = self.bn2(out)
    out = self.relu(out)   # no `out += identity`
    return out

def cut_skip_connection(block):
    block.forward = types.MethodType(forward_no_skip, block)
    block.downsample = None   # the 1x1 conv+BN shortcut becomes dead code with no skip-add left to feed;
                              # remove it so it doesn't sit in model.parameters() with no gradient (autograd
                              # would otherwise refuse torch.autograd.grad() over an unused parameter)

control_block = 8   # last, non-downsample block - matches the paper's Variant 2

models = {
    'Original': copy.deepcopy(base_model),
    'Variant 1 (cut block %d, critical)' % critical_block: copy.deepcopy(base_model),
    'Variant 2 (cut block %d, control)' % control_block: copy.deepcopy(base_model),
}
names_list = list(models.keys())

cut_skip_connection(get_blocks(models[names_list[1]])[critical_block - 1])
cut_skip_connection(get_blocks(models[names_list[2]])[control_block - 1])

for name, m in models.items():
    m.to(**setup)
    with torch.no_grad():
        out = m(torch.randn(2, 3, 32, 32, **setup))
    print(name, "-> forward OK, output shape", tuple(out.shape))

## 4. Huấn luyện

15 epoch mỗi biến thể (SGD momentum, step LR decay) — đủ để so sánh accuracy tương đối giữa 3 biến thể (không phải để đạt SOTA). Dữ liệu CIFAR-10 dùng lại `part1_dlg_lenet/data`.

In [ ]:
EPOCHS = 15
defs = inversefed.training_strategy('conservative', lr=0.1, epochs=EPOCHS)
defs.batch_size = 128
loss_fn, trainloader, validloader = inversefed.construct_dataloaders(
    'CIFAR10', defs, data_path="../part1_dlg_lenet/data")


def train_model(model, epochs):
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[int(epochs * 0.5), int(epochs * 0.75)], gamma=0.1)
    for epoch in range(epochs):
        model.train()
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(**setup), targets.to(device=setup['device'])
            optimizer.zero_grad()
            loss = F.cross_entropy(model(inputs), targets)
            loss.backward()
            optimizer.step()
        scheduler.step()
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, targets in validloader:
                inputs, targets = inputs.to(**setup), targets.to(device=setup['device'])
                correct += (model(inputs).argmax(1) == targets).sum().item()
                total += targets.size(0)
        acc = correct / total
        if epoch % 5 == 0 or epoch == epochs - 1:
            print("  epoch %2d | train loss %.3f | val acc %.3f" % (epoch, loss.item(), acc))
    return acc


accuracies = {}
for name, model in models.items():
    print("Training:", name)
    t0 = time.time()
    accuracies[name] = train_model(model, EPOCHS)
    print("  -> final val accuracy %.3f (%.1fs)\n" % (accuracies[name], time.time() - t0))
    model.eval()

## 5. Attack — Inverting Gradients (Part 2) trên cả 3 biến thể

Dùng **đúng `inversefed.GradientReconstructor`** và **đúng `config` đầy đủ ở `part2.ipynb` mục 4** (24 000 iteration × 2 restart, `filter='median'`), cùng một ảnh CIFAR-10 (validation set, index 0) cho cả 3 biến thể. Ngân sách nhỏ hơn (3 000 iteration, 1 restart) đã thử trước và cho **cả 3 biến thể ra nhiễu thuần túy, không phân biệt được** — model đã train (~86% accuracy) khó attack hơn hẳn model random-init mà các notebook khác trong project dùng, nên cần ngân sách đầy đủ. Mỗi lần attack ở ngân sách này mất khoảng 15-25 phút; cả mục này có thể chạy 45-80 phút. Nếu Rank Analysis dự đoán đúng, **Variant 1** (cắt block critical) phải cho MSE cao hơn / PSNR thấp hơn rõ rệt so với **Original** và **Variant 2**.

In [ ]:
dm = torch.as_tensor(inversefed.consts.cifar10_mean, **setup)[:, None, None]
ds = torch.as_tensor(inversefed.consts.cifar10_std, **setup)[:, None, None]

img, label = validloader.dataset[0]
labels = torch.as_tensor((label,), device=setup['device'])
ground_truth = img.to(**setup).unsqueeze(0)

config = dict(signed=True, boxed=True, cost_fn='sim', indices='top10', weights='equal',
             lr=0.1, optim='adam', restarts=2, max_iterations=24_000, total_variation=1e-6,
             init='randn', filter='median', lr_decay=True, scoring_choice='loss')

attack_results = {}
for name, model in models.items():
    model.eval(); model.zero_grad()
    target_loss, _, _ = loss_fn(model(ground_truth), labels)
    input_gradient = [g.detach() for g in torch.autograd.grad(target_loss, model.parameters())]

    rec_machine = inversefed.GradientReconstructor(model, (dm, ds), config, num_images=1)
    t0 = time.time()
    output, stats = rec_machine.reconstruct(input_gradient, labels, img_shape=(3, 32, 32))
    elapsed = time.time() - t0

    mse = (output.detach() - ground_truth).pow(2).mean().item()
    psnr = inversefed.metrics.psnr(output, ground_truth, factor=1 / ds)
    attack_results[name] = dict(mse=mse, psnr=psnr, image=output.detach().cpu(), time=elapsed)
    print("%-40s MSE %.4f | PSNR %.2f dB | %.1fs" % (name, mse, psnr, elapsed))

## 6. Kết quả

Hàng đầu: ground truth + ảnh reconstruct từ 3 biến thể. Bảng dưới đối chiếu **RA-i dự đoán** (mục 2), **accuracy** (mục 4, để xác nhận "không đổi accuracy") và **chất lượng reconstruct thật** (mục 5) — đúng cấu trúc Figure 4 của paper.

In [ ]:
fig, axes = plt.subplots(1, 1 + len(models), figsize=(4 * (1 + len(models)), 4.2))
def to_img(t):
    return (t.clone().detach().to(ds.device) * ds + dm).clamp(0, 1)[0].permute(1, 2, 0).cpu()

axes[0].imshow(to_img(ground_truth)); axes[0].set_title("ground truth"); axes[0].axis('off')
for ax, (name, res) in zip(axes[1:], attack_results.items()):
    ax.imshow(to_img(res['image'])); ax.axis('off')
    ax.set_title("%s\nMSE %.3f | PSNR %.1f dB" % (name, res['mse'], res['psnr']), fontsize=8)
plt.tight_layout()
plt.show()

print("%-40s %10s %10s %10s %12s" % ("variant", "RA-i", "val acc", "MSE", "PSNR (dB)"))
ra_by_block = {1: ra_values[1], critical_block: ra_values[critical_block], control_block: ra_values[control_block]}
block_of = {names_list[0]: None, names_list[1]: critical_block, names_list[2]: control_block}
for name in names_list:
    ra_str = "%d" % ra_values[block_of[name]] if block_of[name] else "-"
    r = attack_results[name]
    print("%-40s %10s %10.3f %10.4f %12.2f" % (name, ra_str, accuracies[name], r['mse'], r['psnr']))

## 7. So sánh với paper (Figure 4) và với Part 3 chính (LeNet)

**So với paper.** Paper báo cáo: cắt skip-connection của block critical (block 3) làm MSE reconstruction **tăng gần 3 lần** so với bản gốc, trong khi cắt block không quan trọng (block 8) gần như không đổi — và cả 3 biến thể **đạt accuracy tương đương nhau** (hai biến thể còn "hơi tốt hơn" một chút). Notebook này verify với ngân sách nhỏ hơn (15 epoch thay vì 200, 1 lần thử thay vì lấy trung bình), nên số liệu tuyệt đối sẽ khác, nhưng nếu Rank Analysis + attack hoạt động đúng, **xu hướng phải giống**: Variant 1 tệ hơn rõ rệt, Variant 2 gần như không đổi, accuracy cả 3 xấp xỉ nhau.

**So với `part3.ipynb`/`part3_batch2.ipynb` (LeNet, thuật toán `r_gap`).** Đây là điểm khác biệt quan trọng nhất của notebook này: nó **không chứng minh gì về bản thân thuật toán `r_gap`** trên kiến trúc phức tạp — nó cho thấy R-GAP **đóng góp một công cụ phân tích** (Rank Analysis) hữu ích ngay cả khi thuật toán đệ quy của nó không chạy được trực tiếp trên kiến trúc đó. Đây cũng chính xác là điều paper tự nhận trong phần Discussion: *"Our rank analysis can be applied to the analysis of both closed-form and optimization-based attacks such as DLG"* — Rank Analysis là đóng góp tổng quát hơn bản thân thuật toán `r_gap`.